In [ ]:
# --- paths come from human/config.py (auto-inserted by fix_notebooks.py) ---
import sys; sys.path.append('..')
from config import HUMAN_BASE


# Figure 3 — GTEx TF State Analysis
Complete self-contained notebook from raw data to final figures.

**Outputs:**
- `Fig3a_consistency_violin.pdf` — master reg vs other TFs
- `Fig3b_tissue_clustering.pdf` — hierarchical clustering dendrogram
- `Fig3c_nstates_bar.pdf` — n_states distribution


## Cell 1 — Imports and settings

In [ ]:
import os, math, itertools, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import openpyxl
from collections import defaultdict, Counter
from itertools import combinations
from scipy.stats import ttest_ind, gaussian_kde, mannwhitneyu
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import pdist, squareform
from statsmodels.stats.multitest import multipletests
from kneed import KneeLocator

warnings.filterwarnings('ignore')
np.seterr(all='ignore')

# ── File paths — update these to your local paths ────────────────
LAMBERT_XLSX = f"{HUMAN_BASE}/GTEx_v11/1-s2.0-S0092867418301065-mmc2.xlsx"
GTEx_TPM     = f"{HUMAN_BASE}/GTEx_v11/GTEx_Analysis_2025-08-22_v11_RNASeQCv2.4.3_gene_tpm.gct"
GTEx_ATTRS   = f"{HUMAN_BASE}/GTEx_v11/GTEx_Analysis_v11_Annotations_SampleAttributesDS.txt"

# ── Parameters ────────────────────────────────────────────────────
MIN_SAMPLES           = 10
MIN_EXPR              = 1.0
P_VALUE_CUTOFF        = 0.05
MAX_DONORS_PER_TISSUE = 50

os.makedirs('gtex_tf_output', exist_ok=True)
os.makedirs('gtex_grn_input/GMM_figures', exist_ok=True)
print('Ready.')

## Cell 2 — Load Lambert TF list

In [ ]:
wb   = openpyxl.load_workbook(LAMBERT_XLSX, read_only=True)
ws   = wb['Table S1. Related to Figure 1B']
rows = list(ws.iter_rows(values_only=True))
lambert_tfs     = {r[0]: r[1] for r in rows[1:] if r[3] == 'Yes' and r[0] is not None}
lambert_ensembl = set(lambert_tfs.keys())
print(f'Confirmed TFs: {len(lambert_tfs)}')

## Cell 3 — Load GTEx sample attributes

In [ ]:
attrs      = pd.read_csv(GTEx_ATTRS, sep='\t', index_col=0, low_memory=False)
tissue_map = attrs['SMTSD'].dropna().to_dict()
tissues    = attrs['SMTSD'].dropna().unique()
print(f'{len(attrs)} samples, {len(tissues)} unique tissues')

## Cell 4 — Load GTEx TPM matrix and compute per-tissue medians

In [ ]:
print('Loading TPM matrix (may take a few minutes)...')
tpm        = pd.read_csv(GTEx_TPM, sep='\t', skiprows=2, index_col=0, low_memory=False)
gene_names = tpm['Description'].to_dict()
tpm        = tpm.drop(columns=['Description'])
print(f'Shape: {tpm.shape}')

common_samples = [s for s in tpm.columns if s in tissue_map]
tpm            = tpm[common_samples]
print(f'Samples with tissue annotation: {len(common_samples)}')

print('Computing per-tissue medians...')
tissue_medians = {}
for tissue in sorted(tissues):
    samps = [s for s in tpm.columns if tissue_map.get(s) == tissue]
    if len(samps) >= MIN_SAMPLES:
        tissue_medians[tissue] = tpm[samps].median(axis=1)

tissue_df = pd.DataFrame(tissue_medians)
print(f'Tissues with >= {MIN_SAMPLES} samples: {tissue_df.shape[1]}')

## Cell 5 — Filter for expressed TF genes and compute variance

In [ ]:
tf_rows = [(gid, lambert_tfs[gid.split('.')[0]])
           for gid in tissue_df.index if gid.split('.')[0] in lambert_ensembl]

tf_df       = tissue_df.loc[[r[0] for r in tf_rows]].copy()
tf_df.index = [r[1] for r in tf_rows]
tf_df       = tf_df[~tf_df.index.duplicated(keep='first')]
tf_df       = tf_df[tf_df.max(axis=1) >= MIN_EXPR]
print(f'Expressed TF genes: {len(tf_df)}')

tf_log = np.log2(tf_df + 1)
tf_var = tf_log.var(axis=1).sort_values(ascending=False)
print(f'Variance range: {tf_var.min():.3f} – {tf_var.max():.3f}')

## Cell 6 — Elbow-based TF selection (high-variance TFs)

In [ ]:
var_sorted = tf_var.sort_values(ascending=False)
x = np.arange(len(var_sorted))
y = var_sorted.values

kneel     = KneeLocator(x, y, curve='convex', direction='decreasing', interp_method='polynomial')
elbow_idx = kneel.knee
elbow_var = var_sorted.iloc[elbow_idx]

print(f'Elbow at rank {elbow_idx+1}, variance = {elbow_var:.3f}')
print(f'High-variance TFs: {(tf_var >= elbow_var).sum()}')

elbow_genes  = var_sorted.iloc[:elbow_idx+1].index.tolist()
Column_order = elbow_genes
print(f'Column_order: {len(Column_order)} TFs')

## Cell 7 — Build Samples_Dic and mRNA_steady_states

In [ ]:
def tissue_to_prefix(t):
    return t.replace(' - ', '__').replace(' ', '_').replace('(','').replace(')','')

orig_Samples_Dic = {}
for sid in [s for s in tpm.columns if s in tissue_map]:
    orig_Samples_Dic.setdefault(tissue_map[sid], []).append(sid)
orig_Samples_Dic = {t: s for t, s in orig_Samples_Dic.items() if len(s) >= MIN_SAMPLES}

name_to_id = {v: k for k, v in gene_names.items()}
elbow_ids  = [name_to_id[g] for g in elbow_genes if g in name_to_id]
tpm_tf     = tpm.loc[elbow_ids].copy()
tpm_tf.index = [gene_names[i] for i in elbow_ids]
tpm_tf       = tpm_tf[[s for s in tpm_tf.columns if s in tissue_map]]

Samples_Dic, mRNA_steady_states = {}, {}
for tissue in sorted(orig_Samples_Dic.keys()):
    prefix = tissue_to_prefix(tissue)
    Samples_Dic[prefix] = []
    for rep_n, orig_id in enumerate(orig_Samples_Dic[tissue]):
        new_id = f'{prefix}_rep{rep_n}'
        mRNA_steady_states[new_id] = tpm_tf[orig_id].to_dict()
        Samples_Dic[prefix].append(new_id)

tissues_sorted = sorted(Samples_Dic.keys())
print(f'Tissues: {len(Samples_Dic)},  Total samples: {len(mRNA_steady_states)}')

## Cell 8 — State calling helper functions

In [ ]:
rng = np.random.default_rng(seed=42)

def cliff_delta_fast(g1, g2):
    g1, g2  = np.asarray(g1, dtype=float), np.asarray(g2, dtype=float)
    greater = np.sum(g1[:, None] > g2[None, :])
    less    = np.sum(g1[:, None] < g2[None, :])
    return abs((greater - less) / (len(g1) * len(g2)))

def hodges_lehmann(x, y):
    return abs(np.median(np.subtract.outer(
        np.asarray(x, dtype=float), np.asarray(y, dtype=float))))

def composite_score(x, y):
    if len(x) == 0 or len(y) == 0: return math.nan
    between    = cliff_delta_fast(x, y) * hodges_lehmann(x, y)
    pooled_std = np.sqrt((np.var(x) + np.var(y)) / 2)
    if pooled_std < 0.01: return between
    snr = hodges_lehmann(x, y) / pooled_std
    return between if snr >= 1.0 else between * snr

def safe_kde(x, **kwargs):
    x = np.asarray(x)
    if x.size < 2 or np.all(x == x.flat[0]): return None
    return gaussian_kde(x, **kwargs)

def compute_all_ad_pairs(groups):
    return pd.DataFrame([
        {'group1': i, 'group2': j, 'p_value': composite_score(g1, g2)}
        for (i, g1), (j, g2) in combinations(enumerate(groups), 2)
    ])

def merge_with_threshold(groups, df_pairs, alpha):
    n = len(groups)
    parent = list(range(n))
    def find(x):
        while parent[x] != x: parent[x] = parent[parent[x]]; x = parent[x]
        return x
    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb: parent[rb] = ra
    for _, row in df_pairs.iterrows():
        if row.p_value <= alpha: union(int(row.group1), int(row.group2))
    comps = defaultdict(list)
    for i in range(n): comps[find(i)].append(i)
    merged, merged_map = [], {}
    for midx, idxs in enumerate(comps.values()):
        merged.append([v for idx in idxs for v in groups[idx]])
        merged_map[midx] = idxs
    return merged, merged_map

def find_elbow_idx_by_cutoff(seq, cutoff):
    if len(seq) < 2: return None, None
    freq     = Counter(seq)
    most_val = max(freq.items(), key=lambda kv: (kv[1], kv[0]))[0]
    drops    = [seq[i] - seq[i+1] for i in range(len(seq)-1)]
    for size in sorted({d for d in drops if d > 0}, reverse=True):
        for i, d in enumerate(drops):
            if d == size and seq[i+1] <= cutoff: return most_val, i+1
    return most_val, None

def teset_and_merge_welch(optimal_clusters, optimal_mapping, alpha=0.01):
    merged = [list(sub) for sub in optimal_clusters]
    try:    merged_map = [list(optimal_mapping[i]) for i in range(len(optimal_clusters))]
    except: merged_map = [list(optimal_mapping[k]) for k in sorted(optimal_mapping.keys())]
    while True:
        n = len(merged)
        if n <= 1: break
        p_mat = np.full((n, n), -np.inf)
        for i in range(n):
            for j in range(i+1, n):
                a = np.asarray(merged[i], dtype=float); a = a[~np.isnan(a)]
                b = np.asarray(merged[j], dtype=float); b = b[~np.isnan(b)]
                if a.size < 2 or b.size < 2: continue
                try:
                    _, p_val = ttest_ind(a, b, equal_var=False)
                    if not np.isnan(p_val): p_mat[i, j] = p_mat[j, i] = p_val
                except: continue
        max_p = np.max(p_mat)
        if not np.isfinite(max_p) or max_p < alpha: break
        i, j = divmod(np.argmax(p_mat), n)
        if i > j: i, j = j, i
        merged[i].extend(merged[j]); merged_map[i].extend(merged_map[j])
        del merged[j]; del merged_map[j]
    return merged, {k: merged_map[k] for k in range(len(merged_map))}

def call_states_for_gene(gene, Samples_Dic, mRNA_steady_states, alpha=P_VALUE_CUTOFF):
    tissues_sorted = sorted(Samples_Dic.keys())
    TPM_per_tissue = []
    for tissue in tissues_sorted:
        vals = [float(mRNA_steady_states[s].get(gene, 0.0)) for s in Samples_Dic[tissue]]
        if len(vals) > MAX_DONORS_PER_TISSUE:
            vals = rng.choice(vals, size=MAX_DONORS_PER_TISSUE, replace=False).tolist()
        vals = [np.log2(v + 1) for v in vals]
        if len(vals) >= 4:
            m, s = np.mean(vals), np.std(vals)
            vals = [v for v in vals if abs(v - m) <= 3 * s]
        TPM_per_tissue.append(vals)

    clean = [v for v in TPM_per_tissue if len(v) >= 2]
    if len(clean) < 2:
        all_v = [v for sub in TPM_per_tissue for v in sub]
        return 1, [float(np.median(all_v)) if all_v else 0.0], {t: 0 for t in tissues_sorted}

    df_pairs   = compute_all_ad_pairs(clean)
    alphas     = np.linspace(df_pairs['p_value'].min(), df_pairs['p_value'].max(), 100, endpoint=False)
    n_clusters = [len(merge_with_threshold(clean, df_pairs, a)[0]) for a in alphas]
    most_val, elbow_i = find_elbow_idx_by_cutoff(n_clusters, 8)

    if elbow_i is None:
        opt_alpha = df_pairs['p_value'].min()
    elif most_val == 1 and n_clusters.count(1) >= len(n_clusters) - 1:
        opt_alpha = alphas[n_clusters.index(1)]
    else:
        opt_alpha = alphas[elbow_i]

    opt_clusters, opt_mapping = merge_with_threshold(clean, df_pairs, opt_alpha)
    filtered = [(c, opt_mapping[i]) for i, c in enumerate(opt_clusters) if len(c) > 1]
    if not filtered:
        all_v = [v for sub in TPM_per_tissue for v in sub]
        return 1, [float(np.median(all_v)) if all_v else 0.0], {t: 0 for t in tissues_sorted}

    opt_clusters, opt_mapping = zip(*filtered)
    opt_clusters = list(opt_clusters)
    opt_mapping  = {i: list(v) for i, v in enumerate(opt_mapping)}
    opt_clusters, opt_mapping = teset_and_merge_welch(opt_clusters, opt_mapping, alpha=alpha)

    order        = np.argsort([np.mean(c) for c in opt_clusters])
    opt_clusters = [opt_clusters[i] for i in order]
    medians      = [float(2**np.median(c) - 1) for c in opt_clusters]

    tissue_states = {}
    for tissue, vals in zip(tissues_sorted, TPM_per_tissue):
        if not vals: tissue_states[tissue] = 0
        else:
            dists = [abs(np.mean(c) - np.mean(vals)) for c in opt_clusters]
            tissue_states[tissue] = int(np.argmin(dists))

    return len(opt_clusters), medians, tissue_states

print('State calling functions loaded.')

## Cell 9 — Run state calling on high-variance TFs

In [ ]:
state_results = {}
print(f'Running state calling for {len(Column_order)} TFs x {len(tissues_sorted)} tissues...')
t0 = time.time()

for gi, gene in enumerate(Column_order):
    n_states, medians, tissue_states = call_states_for_gene(
        gene, Samples_Dic, mRNA_steady_states)
    state_results[gene] = {'n_states': n_states, 'medians': medians,
                            'tissue_states': tissue_states}
    if (gi + 1) % 20 == 0:
        elapsed = time.time() - t0
        eta     = elapsed / (gi+1) * (len(Column_order) - gi - 1)
        print(f'  {gi+1}/{len(Column_order)} ({elapsed:.0f}s, ETA {eta:.0f}s)')

print(f'Done in {time.time()-t0:.0f}s')
dist = Counter(v['n_states'] for v in state_results.values())
for n, cnt in sorted(dist.items()):
    print(f'  {n} states: {cnt} TFs ({cnt/len(Column_order)*100:.1f}%)')

## Cell 10 — State calling on low-variance TFs

In [ ]:
low_var_tfs = [g for g in tf_var.index if g not in set(Column_order)]
print(f'Low-variance TFs: {len(low_var_tfs)}')

lv_ids    = [name_to_id[g] for g in low_var_tfs if g in name_to_id]
tpm_lv    = tpm.loc[lv_ids].copy()
tpm_lv.index = [gene_names[i] for i in lv_ids]
tpm_lv    = tpm_lv[[s for s in tpm_lv.columns if s in tissue_map]]

mRNA_steady_states_lv = {}
for tissue in sorted(orig_Samples_Dic.keys()):
    prefix = tissue_to_prefix(tissue)
    for rep_n, orig_id in enumerate(orig_Samples_Dic[tissue]):
        new_id = f'{prefix}_rep{rep_n}'
        if new_id not in mRNA_steady_states_lv:
            mRNA_steady_states_lv[new_id] = {}
        for gid in lv_ids:
            gsym = gene_names[gid]
            if orig_id in tpm_lv.columns:
                mRNA_steady_states_lv[new_id][gsym] = float(tpm_lv.loc[gsym, orig_id])

print(f'Running state calling for {len(low_var_tfs)} low-variance TFs...')
t0 = time.time()
state_results_lv = {}
for gi, gene in enumerate(low_var_tfs):
    n, med, ts = call_states_for_gene(gene, Samples_Dic, mRNA_steady_states_lv)
    state_results_lv[gene] = {'n_states': n, 'medians': med, 'tissue_states': ts}
    if (gi+1) % 100 == 0:
        print(f'  {gi+1}/{len(low_var_tfs)} ({time.time()-t0:.0f}s)')
print(f'Done in {time.time()-t0:.0f}s')

## Cell 11 — Within-tissue consistency

In [ ]:
def compute_wc(genes, sr, Samples_Dic, mRNA_steady_states, tissues_sorted):
    records = []
    for gene in genes:
        res         = sr[gene]
        n_states    = res['n_states']
        medians_log = [np.log2(m + 1) for m in res['medians']]
        ts          = res['tissue_states']

        tissue_cons = []
        for tissue in tissues_sorted:
            donors = Samples_Dic.get(tissue, [])
            if not donors: continue
            assigned = ts.get(tissue)
            if assigned is None: continue
            donor_vals = np.array([
                np.log2(float(mRNA_steady_states[s].get(gene, 0.0)) + 1)
                for s in donors
            ])
            if len(donor_vals) == 0: continue
            if n_states < 2:
                tissue_cons.append(1.0)
            else:
                consistent = sum(
                    np.argmin([abs(v - m) for m in medians_log]) == assigned
                    for v in donor_vals
                )
                tissue_cons.append(consistent / len(donor_vals))

        records.append({
            'gene':             gene,
            'n_states':         n_states,
            'mean_consistency': np.mean(tissue_cons) if tissue_cons else np.nan,
        })
    return pd.DataFrame(records)

wc_df    = compute_wc(Column_order,  state_results,    Samples_Dic, mRNA_steady_states,    tissues_sorted)
wc_lv_df = compute_wc(low_var_tfs,   state_results_lv, Samples_Dic, mRNA_steady_states_lv, tissues_sorted)

all_wc = pd.concat([
    wc_df[['gene','n_states','mean_consistency']],
    wc_lv_df[['gene','n_states','mean_consistency']],
], ignore_index=True).dropna(subset=['mean_consistency'])

print(f'Total TFs with consistency scores: {len(all_wc)}')
print(f'  n_states=1: {(all_wc["n_states"]==1).sum()}')
print(f'  n_states>=2: {(all_wc["n_states"]>=2).sum()}')
print(all_wc['mean_consistency'].describe().round(3))

## Cell 12 — Master regulator set (60 curated TFs)

In [ ]:
MASTER_REGULATORS_V2 = {
    'Pancreas':  ['PDX1','PTF1A','NKX6-1','PAX4','NKX2-2','NEUROD1'],
    'Liver':     ['HNF4A','FOXA2','FOXA1','HNF1A','HNF1B','CEBPA','NR1H4'],
    'Brain':     ['SOX2','OLIG1','OLIG2','FOXG1','POU3F2','POU3F3',
                  'EMX2','ARX','NKX2-2','NEUROD1'],
    'Heart':     ['NKX2-5','TBX5','HAND2','HAND1','GATA4','MEF2C','TBX20'],
    'Kidney':    ['PAX8','WT1','HNF1B','PAX2','LHX1'],
    'Skin':      ['TP63','KLF4','ZNF750','GRHL1','GRHL2','GRHL3','OVOL1'],
    'Intestine': ['CDX2','CDX1','GATA6','GATA4','HNF4A'],
    'Lung':      ['NKX2-1','SOX17','FOXA2','GATA6'],
    'Blood':     ['GATA1','GATA2','TAL1','IRF4','RORC','TBX21'],
    'Muscle':    ['SRF'],
    'Adipose':   ['PPARG','CEBPA','CEBPB','KLF5'],
    'Testis':    ['DMRT1','SOX9','NR5A1'],
    'Thyroid':   ['PAX8','NKX2-1','FOXE1'],
    'Pituitary': ['POU1F1','PROP1'],
}
all_master_regs = set(tf for tfs in MASTER_REGULATORS_V2.values() for tf in tfs)
assert len(all_master_regs) == 60, f'Expected 60, got {len(all_master_regs)}'

found   = all_master_regs & set(all_wc['gene'])
missing = all_master_regs - set(all_wc['gene'])
print(f'Master regulators: {len(all_master_regs)} unique TFs')
print(f'Found in GTEx set: {len(found)}/60')
if missing: print(f'Missing: {sorted(missing)}')

## Cell 10b — Post-hoc Welch + Holm correction & MR diagnostic plots

Defines `get_state_samples()`, `state_separation_max_p()`, `half_violin()`, `plot_gene_states()`. Runs Welch + Holm QC for all 60 master regulators and saves per-gene diagnostic plots. Inline preview shows one representative per n_states level. Depends on: Cell 7 (`Samples_Dic`, `mRNA_steady_states`), Cell 9 (`state_results`), Cell 12 (`MASTER_REGULATORS_V2`).

In [ ]:
# ── Step 10b: Post-hoc Welch + Holm correction & MR diagnostic plots ──────────
# Depends on: state_results (Cell 9), state_results_lv (Cell 10),
#             Samples_Dic / mRNA_steady_states (Cell 7),
#             mRNA_steady_states_lv (Cell 10),
#             MASTER_REGULATORS_V2 (Cell 12)

# ── 1. Unified lookups across high-var and low-var results ────────────────────
def _lookup_state_result(gene):
    """Return (state_result, mRNA_dict) from whichever pool the gene lives in."""
    if gene in state_results:
        return state_results[gene], mRNA_steady_states
    if gene in state_results_lv:
        return state_results_lv[gene], mRNA_steady_states_lv
    return None, None

# ── 2. Helper: rebuild state_samples from tissue_states ───────────────────────
def get_state_samples(gene, state_result, Samples_Dic, mRNA_ss):
    """
    Reconstruct per-state donor log2-TPM lists using the already-called tissue_states.
    Avoids re-running expensive state-calling; fully consistent with state_results.
    """
    n_states      = state_result['n_states']
    tissue_states = state_result['tissue_states']
    state_samples = [[] for _ in range(n_states)]
    for tissue, donors in Samples_Dic.items():
        si = tissue_states.get(tissue, 0)
        for s in donors:
            v = np.log2(float(mRNA_ss[s].get(gene, 0.0)) + 1)
            state_samples[si].append(v)
    return state_samples

# ── 3. Helper: max Holm-corrected p across all state pairs ────────────────────
def state_separation_max_p(state_samples):
    """Pairwise Welch t-test + Holm correction. Returns max corrected p (worst pair)."""
    n = len(state_samples)
    if n < 2: return float('nan')
    pvals = []
    for i, j in combinations(range(n), 2):
        a = np.asarray(state_samples[i], dtype=float); a = a[~np.isnan(a)]
        b = np.asarray(state_samples[j], dtype=float); b = b[~np.isnan(b)]
        if a.size < 2 or b.size < 2: pvals.append(1.0); continue
        try:
            _, p = ttest_ind(a, b, equal_var=False)
            pvals.append(p if not np.isnan(p) else 1.0)
        except Exception: pvals.append(1.0)
    if not pvals: return float('nan')
    _, pvals_corr, _, _ = multipletests(pvals, method='holm')
    return float(max(pvals_corr))

# ── 4. Diagnostic plot functions ──────────────────────────────────────────────
COLORS = ['#FF0000','#FF7F00','#FFFF00','#00FF00','#0000FF','#4B0082','#8B00FF','#FF00FF']

def half_violin(ax, data, y_pos, color, width=0.4, bw_method='scott'):
    data = np.asarray(data, dtype=float)
    data = data[~np.isnan(data)]
    if len(data) < 2:
        ax.scatter(data, [y_pos]*len(data), color=color, s=20, zorder=3); return
    kde = safe_kde(data, bw_method=bw_method)
    if kde is None:
        ax.scatter(data, [y_pos]*len(data), color=color, s=20, zorder=3); return
    xs = np.linspace(data.min(), data.max(), 300)
    ys = kde(xs)
    ys_norm = ys / ys.max() * width
    ax.fill_between(xs, y_pos, y_pos + ys_norm, color=color, alpha=0.6)
    ax.scatter(data, np.full(len(data), y_pos), color=color, s=12, alpha=0.9, zorder=3)

def plot_gene_states(gene, state_result, Samples_Dic, mRNA_ss,
                     max_p=None, label=None,
                     save_dir='gtex_grn_input/GMM_figures'):
    tissues_sorted_local = sorted(Samples_Dic.keys())
    n_states      = state_result['n_states']
    medians       = state_result['medians']
    tissue_states = state_result['tissue_states']

    # ── Collect raw TPM per tissue, then convert to log2(TPM+1) ──────────────
    all_vals_raw, tissue_vals_raw = [], {}
    for tissue in tissues_sorted_local:
        vals = [float(mRNA_ss[s].get(gene, 0.0)) for s in Samples_Dic[tissue]]
        tissue_vals_raw[tissue] = vals
        all_vals_raw.extend(vals)

    # log2 scale throughout — avoids State-1 KDE spike dominating the y-axis
    tissue_vals = {t: [np.log2(v + 1) for v in vs] for t, vs in tissue_vals_raw.items()}
    all_vals    = np.array([np.log2(v + 1) for v in all_vals_raw])
    medians_log = [np.log2(m + 1) for m in medians]   # convert medians too

    n_t   = len(Samples_Dic)
    fig_h = max(10, n_t * 0.3)
    bot_h = fig_h / 2     # 下半保持当前高度不变
    top_h = 2.5           # 上半单独缩小（按需调，比如 1.8 / 2.0 / 2.5）
    fig, axes = plt.subplots(
        2, 1, figsize=(10, bot_h + top_h), sharex=True,
        gridspec_kw={'height_ratios': [top_h, bot_h]}
    )

    #fig, axes = plt.subplots(2, 1, figsize=(10, max(10, len(Samples_Dic)*0.3)), sharex=True)

    # ── Top: histogram + per-state KDE (log2 scale) ──────────────────────────
    axes[0].hist(all_vals, bins=80, density=True, alpha=0.3, color='gray')
    for si, med_log in enumerate(medians_log):
        sv = np.array([v for t in tissues_sorted_local for v in tissue_vals[t]
                       if tissue_states[t] == si])
        if len(sv) < 2:
            axes[0].axvline(med_log, color=COLORS[si], linewidth=2,
                            label=f'State {si+1} (median={medians[si]:.1f} TPM)')
        else:
            kde = safe_kde(sv)
            if kde is not None:
                xs = np.linspace(sv.min(), sv.max(), 300)
                axes[0].plot(xs, len(sv)/len(all_vals)*kde(xs), color=COLORS[si],
                             linewidth=2,
                             label=f'State {si+1} (median={medians[si]:.1f} TPM)')

    title = fr'Gene expression stable states for $\it{{{gene}}}$ (n_states={n_states})'
    if label: title += f'  [{label}]'
    axes[0].set_title(title, fontsize=12)
    axes[0].set_ylabel(r'Density ($\log_2$ scale)', fontsize=11)

    annot = f'n = {len(all_vals)} GTEx samples'
    if max_p is not None and not np.isnan(max_p):
        annot += f'\nMax Holm-corrected p = {max_p:.2e}'
    axes[0].text(0.97, 0.95, annot, ha='right', va='top',
                 transform=axes[0].transAxes, fontsize=9)

    axes[0].legend(fontsize=9, loc='upper right', bbox_to_anchor=(1, 0.78))
    axes[0].tick_params(axis='x', labelbottom=False)

    # ── Bottom: half-violin per tissue (log2 scale) ───────────────────────────
    tissue_order = sorted(tissues_sorted_local,
                          key=lambda t: (tissue_states[t], np.mean(tissue_vals[t])))
    for y_pos, tissue in enumerate(tissue_order):
        half_violin(axes[1], tissue_vals[tissue], y_pos,
                    COLORS[tissue_states[tissue]], width=0.8)

    def make_short(t):
        if '__' in t:
            parts = t.split('__')
            return f'{parts[0][:8]}..{parts[-1][:18]}'
        return t[:25]

    axes[1].set_yticks(range(len(tissue_order)))
    axes[1].set_yticklabels([make_short(t) for t in tissue_order], fontsize=6)
    axes[1].set_xlabel(r'$\log_2(\mathrm{TPM} + 1)$', fontsize=11)
    axes[1].set_ylabel('Tissue (donors shown)', fontsize=11)

    x_max = max(v for vs in tissue_vals.values() for v in vs)
    x_max_ceil = np.ceil(x_max)
    step = max(1, round((x_max_ceil) / 5))
    x_ticks = np.arange(0, x_max_ceil, step)
    if x_ticks[-1] < x_max_ceil:
        x_ticks = np.append(x_ticks, x_ticks[-1] + step)
    axes[0].set_xlim(-0.2, x_max_ceil)
    axes[1].set_xlim(-0.2, x_max_ceil)
    axes[1].set_xticks(x_ticks)

    plt.tight_layout()
    plt.savefig(f'{save_dir}/{gene}_states.jpg', dpi=150, bbox_inches='tight')
    plt.close()

print('Diagnostic plot functions ready.')

# ── 5. Plot ALL 60 curated MRs (high-var + low-var) ──────────────────────────
from IPython.display import Image, display

MASTER_REGS_FLAT = set(tf for tfs in MASTER_REGULATORS_V2.values() for tf in tfs)

mr_genes_present, mr_genes_missing = [], []
for g in MASTER_REGS_FLAT:
    (mr_genes_present if _lookup_state_result(g)[0] is not None else mr_genes_missing).append(g)

print(f'MRs found: {len(mr_genes_present)}/60  |  missing: {len(mr_genes_missing)}')
if mr_genes_missing:
    print(f'  Not in GTEx / name mismatch: {sorted(mr_genes_missing)}')

mr_max_p = {}
for gene in mr_genes_present:
    sr, mss = _lookup_state_result(gene)
    state_samples = get_state_samples(gene, sr, Samples_Dic, mss)
    n     = sr['n_states']
    max_p = state_separation_max_p(state_samples) if n >= 2 else float('nan')
    mr_max_p[gene] = max_p
    plot_gene_states(gene, sr, Samples_Dic, mss, max_p=max_p, label='MR')

print(f'Saved {len(mr_genes_present)} diagnostic plots to gtex_grn_input/GMM_figures/')

# ── 6. Optional: extra genes to plot + preview (set [] to skip) ──────────────
EXTRA_PREVIEW_GENES = []   # e.g. ['HNF4A', 'FOXA2']

for gene in EXTRA_PREVIEW_GENES:
    sr, mss = _lookup_state_result(gene)
    if sr is None:
        print(f'  {gene}: not found in state_results or state_results_lv, skipping')
        continue
    state_samples = get_state_samples(gene, sr, Samples_Dic, mss)
    n     = sr['n_states']
    max_p = state_separation_max_p(state_samples) if n >= 2 else float('nan')
    plot_gene_states(gene, sr, Samples_Dic, mss, max_p=max_p, label='extra')
    mr_max_p[gene] = max_p

# ── 7. Inline preview ────────────────────────────────────────────────────────
# Shows: up to 2 examples for n_states=1, 1 example per higher n_states value.
# Priority within each bucket: MR > extra > other.
# Safe: only previews genes that were actually plotted (i.e. found in state_results).

all_plotted = mr_genes_present + [g for g in EXTRA_PREVIEW_GENES
                                   if _lookup_state_result(g)[0] is not None]

by_n = {}
for g in all_plotted:
    n = _lookup_state_result(g)[0]['n_states']
    by_n.setdefault(n, []).append(g)

preview_list = []
for n in sorted(by_n):
    quota = 2 if n == 1 else 1
    preview_list.extend(by_n[n][:quota])

for gene in preview_list:
    sr, _ = _lookup_state_result(gene)
    n      = sr['n_states']
    tag    = 'MR' if gene in MASTER_REGS_FLAT else ('extra' if gene in EXTRA_PREVIEW_GENES else '')
    pool   = 'high-var' if gene in state_results else 'low-var'
    print(f'--- n_states={n}  {gene}  [{tag}|{pool}] ---')
    display(Image(f'gtex_grn_input/GMM_figures/{gene}_states.jpg', width=700))

## Cell 13 — Figure 3a: within-tissue consistency violin

In [ ]:
# ────────────────────────────────────────────────────────────────────
# Figure 2a: Within-tissue consistency — master regulators vs other TFs
# Cell Systems format: Arial 7pt, single-column, RGB, ≥300 dpi
# ────────────────────────────────────────────────────────────────────
import math
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

# Cell Press global style
mpl.rcParams['font.family']     = 'Arial'
mpl.rcParams['font.size']       = 7
mpl.rcParams['axes.linewidth']  = 0.8
mpl.rcParams['xtick.major.width'] = 0.8
mpl.rcParams['ytick.major.width'] = 0.8
mpl.rcParams['xtick.major.size']  = 3
mpl.rcParams['ytick.major.size']  = 3
mpl.rcParams['pdf.fonttype']    = 42
mpl.rcParams['ps.fonttype']     = 42

# ── Data subset (n_states >= 2 only) ────────────────────────────────
all_wc_multi = all_wc[all_wc['n_states'] >= 2]
master_cons = all_wc_multi[all_wc_multi['gene'].isin(all_master_regs)]['mean_consistency'].dropna()
other_cons  = all_wc_multi[~all_wc_multi['gene'].isin(all_master_regs)]['mean_consistency'].dropna()

_, p = mannwhitneyu(master_cons, other_cons, alternative='greater')

print(f'n_states>=2: {len(all_wc_multi)} TFs ({len(all_wc)-len(all_wc_multi)} single-state excluded)')
print(f'Master regs (n={len(master_cons)}): mean={master_cons.mean():.3f}, median={np.median(master_cons):.3f}')
print(f'Other TFs   (n={len(other_cons)}):  mean={other_cons.mean():.3f}, median={np.median(other_cons):.3f}')
print(f'Mann-Whitney p = {p:.4e}')

# ── Plot ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7.5, 9.0))

# Colorblind-friendly palette
color_master = '#D55E00'  # vermillion
color_other  = '#0072B2'  # blue

parts = ax.violinplot(
    [master_cons.values, other_cons.values],
    positions=[0, 1],
    showmedians=True,
    showextrema=True,
    widths=0.7,
)
parts['bodies'][0].set_facecolor(color_master)
parts['bodies'][1].set_facecolor(color_other)
for pc in parts['bodies']:
    pc.set_alpha(0.65)
    pc.set_edgecolor('none')

# Median bar: white inside violin
parts['cmedians'].set_color('white')
parts['cmedians'].set_linewidth(1.2)
for key in ('cmaxes', 'cmins', 'cbars'):
    parts[key].set_color('#444444')
    parts[key].set_linewidth(0.8)

# Individual data points — both groups
np.random.seed(42)
# Master regulators (n=60): larger, more opaque
jitter_m = np.random.uniform(-0.08, 0.08, len(master_cons))
ax.scatter(jitter_m, master_cons.values,
           color='#8B3A00', s=6, alpha=0.6, zorder=4, edgecolors='none')
# Other TFs (n=1393): smaller, more transparent
jitter_o = np.random.uniform(-0.08, 0.08, len(other_cons))
ax.scatter(1 + jitter_o, other_cons.values,
           color='#003D5B', s=1.5, alpha=0.12, zorder=4, edgecolors='none')

# Axes
ax.set_xticks([0, 1])
ax.set_xticklabels([
    f'Master\nregulators\n(n={len(master_cons)})',
    f'Other TFs\n(n={len(other_cons)})',
])
ax.set_ylabel('Within-tissue consistency')
ax.set_ylim(0, 1.15)
ax.set_yticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])

# Significance annotation — Cell-style scientific notation
exp = int(math.floor(math.log10(abs(p))))
coef = p / 10**exp
sup_map = {'0':'⁰','1':'¹','2':'²','3':'³','4':'⁴',
           '5':'⁵','6':'⁶','7':'⁷','8':'⁸','9':'⁹','-':'⁻'}
p_str = f'p = {coef:.1f} × 10' + ''.join(sup_map[c] for c in str(exp))

y_ann = 1.08
ax.plot([0, 1], [y_ann, y_ann], color='black', linewidth=0.8)
ax.text(0.5, y_ann + 0.01, p_str, ha='center', va='bottom', fontsize=6)

# Spines
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Gridlines
ax.yaxis.grid(True, alpha=0.25, linestyle='-', linewidth=0.4)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig('gtex_tf_output/Fig2a_consistency_violin.pdf', bbox_inches='tight', dpi=300)
plt.savefig('gtex_tf_output/Fig2a_consistency_violin.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: Fig2a')

## Cell 14 — Figure 3b: tissue clustering dendrogram

In [ ]:
# ── State assignment matrix (n_states>=2 TFs only) ────────────────
informative_tfs = [g for g in Column_order if state_results[g]['n_states'] >= 2]
print(f'Informative TFs: {len(informative_tfs)}')

state_matrix = pd.DataFrame(index=tissues_sorted, columns=informative_tfs, dtype=float)
for gene in informative_tfs:
    ts = state_results[gene]['tissue_states']
    for tissue in tissues_sorted:
        state_matrix.loc[tissue, gene] = ts.get(tissue, 0)

# ── Tissue groups ────────────────────────────────────────────────
TISSUE_GROUPS = {
    'Brain':     ['Brain'],
    'Colon':     ['Sigmoid', 'Transverse'],
    'Skin':      ['Sun_Exposed', 'Not_Sun_Exposed'],
    'Artery':    ['Tibial', 'Coronary', 'Aorta'],
    'Heart':     ['Left_Ventricle', 'Atrial_Appendage'],
    'Esophagus': ['Mucosa', 'Muscularis', 'Gastroesophageal'],
    'Adipose':   ['Subcutaneous', 'Visceral'],
    'Cervix':    ['Ectocervix', 'Endocervix'],
    'Kidney':    ['Kidney'],
}
LINEAGE_COLORS = {
    'Brain':'#4472C4','Colon':'#ED7D31','Skin':'#FFC000',
    'Artery':'#FF0000','Heart':'#C00000','Esophagus':'#7030A0',
    'Adipose':'#00B050','Cervix':'#FF69B4','Kidney':'#00B0F0','Other':'#AAAAAA',
}

def get_lineage(tissue):
    for lin, kws in TISSUE_GROUPS.items():
        if any(kw.lower() in tissue.lower() for kw in kws): return lin
    return 'Other'

def short_name(t):
    t = t.replace('Brain__','Brain_')
    parts = t.split('__')
    return parts[-1][:28] if len(parts)>1 else t[:28]

tissue_lineages = [get_lineage(t) for t in tissues_sorted]
short_names     = [short_name(t) for t in tissues_sorted]

# ── Clustering ────────────────────────────────────────────────────
dist_matrix = pd.DataFrame(
    squareform(pdist(state_matrix.values, metric='hamming')),
    index=tissues_sorted, columns=tissues_sorted
)
Z = linkage(pdist(state_matrix.values, metric='hamming'), method='ward')

fig, ax = plt.subplots(figsize=(5.5, len(tissues_sorted)*0.22+1.5))
dn = dendrogram(Z, labels=short_names, orientation='left', ax=ax,
                leaf_font_size=6.5, color_threshold=0, above_threshold_color='#666666')

for label, leaf_idx in zip(ax.get_yticklabels(), range(len(dn['leaves']))):
    tissue  = tissues_sorted[dn['leaves'][leaf_idx]]
    label.set_color(LINEAGE_COLORS[get_lineage(tissue)])

ax.set_xlabel('Hamming distance', fontsize=10)
ax.set_title(f'Tissue clustering by TF state assignment\n({len(informative_tfs)} informative TFs)', fontsize=10)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

legend_els = [mpatches.Patch(facecolor=LINEAGE_COLORS[l], label=l)
              for l in LINEAGE_COLORS if l in tissue_lineages and l != 'Other']
ax.legend(handles=legend_els, loc='lower left', fontsize=7, framealpha=0.8)

plt.tight_layout()
plt.savefig('gtex_tf_output/Fig3b_tissue_clustering.pdf', bbox_inches='tight')
plt.savefig('gtex_tf_output/Fig3b_tissue_clustering.png', dpi=300, bbox_inches='tight')
plt.show()

# ── Within/between Hamming table ─────────────────────────────────
print(f'{"Group":<12} {"Within":>8} {"Between":>8} {"Ratio":>8} {"n":>4}')
print('-' * 44)
ratios = {}
for lin in sorted(LINEAGE_COLORS):
    if lin == 'Other': continue
    ing  = [t for t in tissues_sorted if get_lineage(t) == lin]
    outg = [t for t in tissues_sorted if get_lineage(t) != lin]
    if len(ing) < 2: continue
    wv = dist_matrix.loc[ing, ing].values
    within  = wv[np.triu_indices(len(ing), k=1)].mean()
    between = dist_matrix.loc[ing, outg].values.mean()
    ratios[lin] = within/between
    print(f'{lin:<12} {within:>8.3f} {between:>8.3f} {within/between:>8.3f} {len(ing):>4}')

print(f'\nAll {sum(r<1 for r in ratios.values())}/{len(ratios)} groups: within < between')
print(f'Most coherent: {min(ratios,key=ratios.get)} (ratio={min(ratios.values()):.3f})')
print('Saved: Fig3b')

## Cell 15 — Figure 3c: n_states distribution bar chart

In [ ]:
n_states_counts = all_wc['n_states'].value_counts().sort_index()
total   = len(all_wc)
multi   = (all_wc['n_states'] >= 2).sum()
mean_st = all_wc['n_states'].mean()

print(f'Total expressed TFs: {total}')
print(f'Multi-state (>=2):   {multi} ({multi/total*100:.1f}%)')
print(f'Mean n_states:       {mean_st:.2f}')

# ── Pie chart ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 5))

states = n_states_counts.index.tolist()
counts = n_states_counts.values.tolist()

# color: n=1 gray, n=2 lightest blue, increasing saturation
base_colors = ['#AAAAAA'] + [
    plt.cm.Blues(0.3 + 0.7 * i / (len(states) - 2))
    for i in range(len(states) - 1)
]

wedges, texts, autotexts = ax.pie(
    counts,
    labels=[f'{s} state{"s" if s > 1 else ""}' for s in states],
    colors=base_colors,
    autopct=lambda pct: f'{pct:.1f}%' if pct >= 3 else '',
    startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=1.2),
    pctdistance=0.75,
)

for t in texts:
    t.set_fontsize(9)
for at in autotexts:
    at.set_fontsize(8)

ax.set_title(
    f'Discrete expression states\nacross {total} expressed TFs (GTEx V11, 60 tissues)\n'
    f'{multi/total*100:.1f}% multi-state  |  mean = {mean_st:.1f} states',
    fontsize=10
)

plt.tight_layout()
plt.savefig('gtex_tf_output/Fig3c_nstates_pie.pdf', bbox_inches='tight')
plt.savefig('gtex_tf_output/Fig3c_nstates_pie.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: Fig3c')
print(n_states_counts)

## Cell 16 — Summary: all numbers for paper

In [ ]:
print('═'*55)
print('NUMBERS FOR PAPER')
print('═'*55)
all_wc_multi = all_wc[all_wc['n_states'] >= 2]
master_cons  = all_wc_multi[all_wc_multi['gene'].isin(all_master_regs)]['mean_consistency'].dropna()
other_cons   = all_wc_multi[~all_wc_multi['gene'].isin(all_master_regs)]['mean_consistency'].dropna()
_, p         = mannwhitneyu(master_cons, other_cons, alternative='greater')

print(f'Expressed TFs:              {len(all_wc)}')
print(f'Multi-state (>=2):          {len(all_wc_multi)} ({len(all_wc_multi)/len(all_wc)*100:.1f}%)')
print(f'Mean n_states:              {all_wc["n_states"].mean():.1f}')
print()
print(f'Master regs (n_states>=2):  n={len(master_cons)}, mean={master_cons.mean():.3f}')
print(f'Other TFs   (n_states>=2):  n={len(other_cons)},  mean={other_cons.mean():.3f}')
print(f'Mann-Whitney p:             {p:.4e}')
print()
print('Tissue Hamming ratios:')
for lin in sorted(ratios):
    print(f'  {lin:<12}: {ratios[lin]:.3f}')

## Cell 17 — Figure 3b (revised): Within/Between Hamming ratio bar chart
Run after Cell 14. Replaces the dendrogram as the main Figure 3b panel.
Depends on: `ratios`, `LINEAGE_COLORS`, `dist_matrix` from Cell 14.

In [ ]:
## ═══════════════════════════════════════════════════════════════════
## Figure 2b: Within/Between Hamming ratio bar chart
## Run after Cell 14 (which computes dist_matrix, ratios)
## ═══════════════════════════════════════════════════════════════════

# ── Fake-tissue negative control ─────────────────────────────────
# Build one fake "lineage" by sampling 1 tissue from each real group.
# Average over 1000 random draws so the result isn't sensitive to a
# single unlucky shuffle.
N_FAKE_DRAWS = 1000
fake_rng     = np.random.default_rng(42)
group_to_tissues = {lin: [t for t in tissues_sorted if get_lineage(t) == lin]
                    for lin in TISSUE_GROUPS}

fake_ratios = []
for _ in range(N_FAKE_DRAWS):
    fake_set = [fake_rng.choice(ts) for ts in group_to_tissues.values() if len(ts) > 0]
    if len(fake_set) < 2: continue
    rest = [t for t in tissues_sorted if t not in fake_set]
    wv   = dist_matrix.loc[fake_set, fake_set].values
    w    = wv[np.triu_indices(len(fake_set), k=1)].mean()
    b    = dist_matrix.loc[fake_set, rest].values.mean()
    fake_ratios.append(w / b)
ratios['Fake'] = float(np.median(fake_ratios))
ratios.pop('Fake', None)
ratios['Control'] = float(np.median(fake_ratios))

# Tissue-semantic palette (perceptually distinguishable, biology-grounded)
LINEAGE_COLORS = {
    'Adipose':   '#F4D27A',  # 米黄 (adipose tissue)
    'Artery':    '#C8455A',  # 鲜红 (arterial blood)
    'Brain':     '#7C6FA8',  # 灰紫 (grey matter)
    'Cervix':    '#D597B6',  # 粉紫 (mucosa)
    'Colon':     '#A57F4B',  # 暖棕 (digestive)
    'Esophagus': '#8B6F4D',  # 棕褐 (esophageal mucosa)
    'Heart':     '#8B2C3A',  # 深红 (myocardium)
    'Kidney':    '#A04848',  # 砖红 (renal cortex)
    'Skin':      '#E89B7B',  # 肤色
    'Fake':      '#BBBBBB',  # 中灰 (negative control)
}

# ── sort by ratio ascending (most coherent first) ─────────────────
# ── sort by ratio descending (most coherent first, at top) ────────
ratio_df = pd.DataFrame([
    {'group': g, 'ratio': r, 'color': LINEAGE_COLORS.get(g, '#AAAAAA')}
    for g, r in ratios.items()
]).sort_values('ratio', ascending=False)

fig, ax = plt.subplots(figsize=(4.5*1.3, 3.5))
bars = ax.barh(
    ratio_df['group'], ratio_df['ratio'],
    color=ratio_df['color'], edgecolor='black', linewidth=0.4, height=0.6
)

# ratio = 1 reference line (within = between)
#ax.axvline(1.0, color='black', linestyle='--', linewidth=1.2, alpha=0.7,
#           label='Within = Between')

max_ratio = ratio_df['ratio'].max()
ax.set_xlabel('Within-group Hamming / Between-group Hamming', fontsize=11)
ax.set_title('Tissue groups cluster by TF state similarity', fontsize=11)
#ax.legend(fontsize=8, loc='lower right', frameon=False)
ax.set_xlim(0, max_ratio * 1.1)   # tighter without labels
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='x', alpha=0.3, linestyle='--')
os.makedirs(f"{HUMAN_BASE}/New_figures", exist_ok=True)
plt.savefig(f"{HUMAN_BASE}/New_figures/Figure 2_new.png", dpi=300)

In [ ]:
# Track which tissue was sampled for each group across all draws.
fake_rng       = np.random.default_rng(42)   # re-seed to replay the same draws
sample_counter = {lin: Counter() for lin in group_to_tissues}
for _ in range(N_FAKE_DRAWS):
    for lin, ts in group_to_tissues.items():
        if len(ts) > 0:
            sample_counter[lin][fake_rng.choice(ts)] += 1
print('Fake tissue (modal pick per group): ' +
      ', '.join(f'{lin}={sample_counter[lin].most_common(1)[0][0]}'
                for lin in sample_counter))

In [ ]:
for lin in sorted(TISSUE_GROUPS):
    members = [t for t in tissues_sorted if get_lineage(t) == lin]
    print(f'{lin:<12} ({len(members)}): ' + ', '.join(members))

other = [t for t in tissues_sorted if get_lineage(t) == 'Other']
print(f'{"Other":<12} ({len(other)}): ' + ', '.join(other))

In [ ]:
# Use lineage groups from Cell 14 (re-use the same TISSUE_GROUPS dict)
TISSUE_GROUPS_FULL = {
    'Brain':     ['Brain'],
    'Colon':     ['Sigmoid', 'Transverse'],
    'Skin':      ['Sun_Exposed', 'Not_Sun_Exposed'],
    'Artery':    ['Tibial', 'Coronary', 'Aorta'],
    'Heart':     ['Left_Ventricle', 'Atrial_Appendage'],
    'Esophagus': ['Mucosa', 'Muscularis', 'Gastroesophageal'],
    'Adipose':   ['Subcutaneous', 'Visceral'],
    'Cervix':    ['Ectocervix', 'Endocervix'],
    'Kidney':    ['Kidney'],
    # rest are 'Other' (one tissue per lineage essentially)
}

def get_lineage(tissue):
    for lin, kws in TISSUE_GROUPS_FULL.items():
        if any(kw.lower() in tissue.lower() for kw in kws): return lin
    return 'Other_' + tissue   # singleton lineage for everything else

def n_lineages_at_highest_state(gene):
    res = state_results.get(gene) or state_results_lv.get(gene)
    if res is None or res['n_states'] < 2: return None
    highest = res['n_states'] - 1
    high_tissues = [t for t, s in res['tissue_states'].items() if s == highest]
    return len(set(get_lineage(t) for t in high_tissues))

# Re-run with lineage-based K_MAX
K_MAX_LIN   = 3      # highest state across at most 3 lineages
CONS_CUTOFF = 0.70

cand_rows = []
for _, row in all_wc.iterrows():
    if row['n_states'] < 2 or row['mean_consistency'] < CONS_CUTOFF: continue
    n_lin = n_lineages_at_highest_state(row['gene'])
    if n_lin is None or not (1 <= n_lin <= K_MAX_LIN): continue
    cand_rows.append({'gene': row['gene'], 'n_states': row['n_states'],
                      'consistency': row['mean_consistency'], 'n_high_lin': n_lin})

cand_df  = pd.DataFrame(cand_rows).sort_values('consistency', ascending=False)
cand_set = set(cand_df['gene'])

overlap        = cand_set & all_master_regs
curated_missed = all_master_regs - cand_set
novel          = cand_set - all_master_regs

print(f'Filters: n_states>=2, highest-state in <= {K_MAX_LIN} lineages, consistency >= {CONS_CUTOFF}')
print(f'{"─"*60}')
print(f'Candidates found        : {len(cand_set)}')
print(f'Overlap w/ curated 60   : {len(overlap)}/60 ({len(overlap)/60*100:.0f}%)')
print(f'Curated TFs missed      : {len(curated_missed)}')
if curated_missed:
    print(f'  {sorted(curated_missed)}')
print(f'Novel candidates        : {len(novel)}')

print('\nWhy each curated TF was missed (under lineage filter):')
for tf in sorted(curated_missed):
    row  = all_wc[all_wc['gene'] == tf]
    if len(row) == 0:
        print(f'  {tf:<10}  not in all_wc'); continue
    r    = row.iloc[0]
    nlin = n_lineages_at_highest_state(tf)
    reasons = []
    if r['n_states'] < 2:                     reasons.append(f'n_states={int(r["n_states"])}')
    if r['mean_consistency'] < CONS_CUTOFF:   reasons.append(f'cons={r["mean_consistency"]:.2f}')
    if nlin is None:                          reasons.append('no state info')
    elif not (1 <= nlin <= K_MAX_LIN):        reasons.append(f'n_lin={nlin}')
    print(f'  {tf:<10}  {", ".join(reasons)}')

In [ ]:
# ===========================================================================
# Export SETIA-style discrete-state output (same convention as build_setia_input)
#   Steady_state_count.txt   gene<TAB>n_states<TAB>tissue:state_label ...
#   discrete_states.txt       rows=tissue, cols=gene, value=log2 state median
#   discrete_states_std.txt   rows=tissue, cols=gene, value=log2 std of assigned state
# Rebuilt from state_results (+ low-var pool); state-calling functions untouched.
# ===========================================================================
GRN_OUT = 'gtex_tf_output'          # change if you want a dedicated folder
os.makedirs(GRN_OUT, exist_ok=True)

# low-var pool may or may not have been run
_SR_LV   = globals().get('state_results_lv', {})
_MRNA_LV = globals().get('mRNA_steady_states_lv', {})

def _pool(gene):
    if gene in state_results:   return state_results[gene],   mRNA_steady_states
    if gene in _SR_LV:          return _SR_LV[gene],           _MRNA_LV
    return None, None

# stable gene order: high-var (Column_order) then low-var
_lv_genes = [g for g in globals().get('low_var_tfs', []) if g in _SR_LV]
all_genes = [g for g in Column_order if g in state_results] + \
            [g for g in _lv_genes if g not in state_results]

samples_sorted = tissues_sorted
med_mat, std_mat, ssc_lines = {}, {}, []

for gene in all_genes:
    res, mrna = _pool(gene)
    if res is None:
        continue
    n  = int(res['n_states'])
    ts = res['tissue_states']

    # group log2 sample values by the FINAL assigned state label
    state_vals = [[] for _ in range(n)]
    for tissue in samples_sorted:
        si = ts.get(tissue, 0)
        if not (0 <= si < n):
            si = 0
        for s in Samples_Dic.get(tissue, []):
            v = np.log2(float(mrna[s].get(gene, 0.0)) + 1)
            state_vals[si].append(v)

    med_log = []
    std_log = []
    for k, sv in enumerate(state_vals):
        if sv:
            med_log.append(float(np.median(sv)))
            std_log.append(float(np.std(sv)))
        else:
            # state with no assigned tissue: fall back to called median (linear -> log2)
            m = res['medians'][k] if k < len(res['medians']) else 0.0
            med_log.append(float(np.log2(m + 1)))
            std_log.append(0.0)

    med_row, std_row, labels = {}, {}, []
    for tissue in samples_sorted:
        si = ts.get(tissue, 0)
        if not (0 <= si < n):
            si = 0
        med_row[tissue] = med_log[si]
        std_row[tissue] = std_log[si]
        labels.append(f'{tissue}:{si}')

    med_mat[gene] = med_row
    std_mat[gene] = std_row
    ssc_lines.append(f'{gene}\t{n}\t' + '\t'.join(labels))

ds   = pd.DataFrame(med_mat, index=samples_sorted).reindex(columns=all_genes)
dstd = pd.DataFrame(std_mat, index=samples_sorted).reindex(columns=all_genes)
ds.index.name = 'tissue'
dstd.index.name = 'tissue'

ds.to_csv(f'{GRN_OUT}/discrete_states.txt',     sep='\t', float_format='%.4f')
dstd.to_csv(f'{GRN_OUT}/discrete_states_std.txt', sep='\t', float_format='%.4f')
with open(f'{GRN_OUT}/Steady_state_count.txt', 'w') as fh:
    fh.write('\n'.join(ssc_lines) + '\n')

print(f'GTEx SETIA output written to {GRN_OUT}/')
print(f'  genes: {len(all_genes)}   tissues: {len(samples_sorted)}')
print(f'  discrete_states.txt     {ds.shape}')
print(f'  discrete_states_std.txt {dstd.shape}')
print(f'  Steady_state_count.txt  {len(ssc_lines)} lines')
